# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the Kilifi County, Kenya FAIR² Model dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, ensuring up-to-date metadata and standardized access.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant-defined dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The FAIR² dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Inspect the record sets and fields in the dataset. All references use their `@id` values as per Croissant best practices.

In [ ]:
# List all record sets with their @id
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"RecordSet name: {getattr(rs, 'name', getattr(rs, '@id', 'unknown'))}, @id: {getattr(rs, '@id', str(rs))}")
        record_set_ids.append(rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', str(rs)))
elif hasattr(metadata, 'record_set'):
    # Sometimes the metadata attribute is 'record_set'
    for rs in metadata.record_set:
        print(f"RecordSet name: {getattr(rs, 'name', getattr(rs, '@id', 'unknown'))}, @id: {getattr(rs, '@id', str(rs))}")
        record_set_ids.append(rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', str(rs)))
else:
    print("No record sets found in metadata.")
# If no record sets are found above, try fallback logic
if not record_set_ids:
    # mlcroissant >=0.3.0 stores record set ids in dataset.record_set_ids
    if hasattr(dataset, 'record_set_ids'):
        record_set_ids = dataset.record_set_ids
        print("RecordSet IDs fetched from dataset object:")
        print(record_set_ids)
    else:
        print("No record sets detected in dataset.")

### Explore fields in each record set
For demonstration, we will print the first found record set's fields and their `@id` (if available).

In [ ]:
# List fields for the first record set (if any)
if len(record_set_ids):
    record_set_id = record_set_ids[0]

    print(f"Fields for RecordSet @id: {record_set_id}")
    sample_fields = []
    try:
        rs_obj = None
        if hasattr(metadata, 'record_sets'):
            for rs in metadata.record_sets:
                rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
                if rs_id == record_set_id:
                    rs_obj = rs
                    break
        elif hasattr(metadata, 'record_set'):
            for rs in metadata.record_set:
                rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
                if rs_id == record_set_id:
                    rs_obj = rs
                    break

        # Try to use the dataset.fields_by_record_set method (if available)
        if hasattr(dataset, 'fields_by_record_set'):
            fields = dataset.fields_by_record_set(record_set_id)
            for field in fields:
                name = getattr(field, 'name', getattr(field, '@id', str(field)))
                id_ = getattr(field, '@id', None)
                print(f"Field: {name} | @id: {id_}")
                sample_fields.append(id_)
        else:
            if rs_obj and 'fields' in rs_obj:
                for field in rs_obj['fields']:
                    name = field.get('name', field.get('@id', ''))
                    id_ = field.get('@id', '')
                    print(f"Field: {name} | @id: {id_}")
                    sample_fields.append(id_)
    except Exception as e:
        print(f"Could not list fields: {e}")
else:
    print("No record set to display fields for.")

## 3. Data Extraction
Load one or more record sets using their `@id`, as per Croissant best practice, into Pandas DataFrames for analysis.

In [ ]:
# You can adjust this list according to actual available record sets. For the example, we use all found record set @ids.
all_dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        all_dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from {rs_id}.")
        if len(df.columns):
            print("Columns (fields):", df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print(f"Could not load data for record set {rs_id}: {e}")

# For simplicity in further analysis, we choose the first found record_set as 'main' for EDA
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Process a numeric field and demonstrate filtering and normalization. We'll try to choose a field containing a score (e.g., GAD-7_total, PHQ-9_total, PSQ_total) if present.

In [ ]:
import numpy as np
# Helper function to pick a numeric field
def select_numeric_field(df):
    # Try to guess common clinical assessment sum scores
    preferred = [
        'GAD-7_total', 'PHQ-9_total', 'PSQ_total', 'gad7_total', 'phq9_total', 'psq_total',
        'score', 'Score', 'TOTAL', 'total', 'Total'
    ]
    for c in preferred:
        if c in df.columns:
            return c
    # Else, pick first float/numeric column
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            return col
    # or string containing numeric
    for col in df.columns:
        if df[col].astype(str).str.isnumeric().any():
            return col
    return None

df_main = all_dataframes[main_record_set_id] if main_record_set_id else None
if df_main is not None:
    numeric_field = select_numeric_field(df_main)
    print(f"Using numeric field: {numeric_field}")
    if numeric_field:
        # Remove missing values and ensure numeric
        df_main[numeric_field] = pd.to_numeric(df_main[numeric_field], errors='coerce')
        threshold = df_main[numeric_field].mean()
        filtered_df = df_main[df_main[numeric_field] > threshold]
        print(f"Filtered rows where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a demographic/category field
        group_fields = ['gender', 'Gender', 'sex', 'village', 'Village', 'age_group', 'AgeGroup', 'marital_status', 'education', 'level_of_education']
        group_field = None
        for g in group_fields:
            if g in df_main.columns:
                group_field = g
                break
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nAverage {numeric_field} by {group_field} (for rows above mean):\n", grouped)
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame to perform EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field before and after normalization, and (if available) by a grouping attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_main is not None and numeric_field:
    plt.figure(figsize=(10,4))
    sns.histplot(df_main[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=15)
        plt.title(f"Distribution of Normalized {numeric_field} (filtered > mean)")
        plt.xlabel(f"{numeric_field} (normalized)")
        plt.show()
    # Boxplot by group
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df_main)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("Visualization skipped due to missing DataFrame or numeric field.")

## 6. Conclusion
This notebook demonstrated how to:

1. **Load** a Croissant-defined dataset and inspect its structure using `mlcroissant`.
2. **Dynamically extract** record sets and fields using their `@id`, in compliance with standards.
3. **Conduct basic EDA**: filter and normalize a survey score field, grouped by a demographic variable.
4. **Visualize** the data distribution for key numeric fields.

**Note:** For further, domain-specific analysis, consult the Croissant schema or accompanying documentation to understand the semantic meaning of each field, and refer to the `@id` in all code for future-proof, standards-compliant processing.